In [34]:
import math

from sklearn.neighbors import KNeighborsClassifier
from tensorflow.keras.datasets import mnist
import numpy as np
from tqdm import tqdm

In [35]:
RANDOM_STATE = 42
EPSILON = 1e-15

In [36]:
from sklearn.model_selection import train_test_split

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_train
)


In [37]:
class LinearLayer:
    def __init__(self, in_features, out_features, random_state: int = 0):
        rng = np.random.RandomState(random_state)

        w_sigma = math.sqrt(2 / (in_features + out_features))
        self.__w = rng.randn(in_features, out_features, ) * w_sigma
        self.__b = np.zeros(out_features)

    def forward(self, x: np.ndarray) -> np.ndarray:
        return np.dot(x, self.__w) + self.__b

    def backward(self, x: np.ndarray, grad: np.ndarray, lr: float) -> np.ndarray:
        reshaped_grad = grad.reshape(1, grad.shape[0])
        reshaped_x = x.reshape(1, x.shape[0])

        de_dx = np.dot(grad, self.__w.T)
        de_dw = np.dot(reshaped_x.T, reshaped_grad).reshape(self.__w.shape)
        de_db = grad

        self.__w -= lr * de_dw
        self.__b -= lr * de_db

        return de_dx


In [38]:
def softmax(data: np.ndarray) -> np.ndarray:
    shifted = data - np.max(data)
    exps = np.exp(shifted)
    return exps / exps.sum()

def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return -np.sum(y_true * np.log2(np.clip(y_pred, EPSILON, 1.0)))

def cross_entropy_gradient(y_pred_softmax: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    return y_pred_softmax - y_true

def flatten(X: np.ndarray) -> np.ndarray:
    return X.reshape(X.shape[0], -1)

def normalize(X: np.ndarray) -> np.ndarray :
    return X / 255.0

In [39]:
from sklearn.base import BaseEstimator, ClassifierMixin

class Model(ClassifierMixin, BaseEstimator):
    def __init__(
        self,
            in_features: int,
            learning_rate: float,
            epochs: int,
            show_progress: bool = False,
            random_state: int = 0
    ):
        self.in_features = in_features
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.show_progress = show_progress
        self.random_state = random_state

        self.__train_errors = []
        self.classes_ = None
        self.__linear = None

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if not isinstance(X, np.ndarray):
            X = np.array([X])

        return np.array(list(map(self.__forward, X)))

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(axis=1)

    def fit(self, X: np.ndarray, y: np.ndarray):
        self.__linear = LinearLayer(
            in_features=self.in_features,
            out_features=10,
            random_state=self.random_state
        )
        self.classes_ = np.unique(y)

        y_probs = np.eye(10)[y]

        with tqdm(total=self.epochs * len(X), desc="Training", disable=not self.show_progress) as pbar:
            for epoch in range(self.epochs):
                for x, y_true in zip(X, y_probs):
                    logits = self.__linear.forward(x)
                    probs = softmax(logits)
                    grad = probs - y_true
                    self.__linear.backward(
                        x,
                        grad,
                        self.learning_rate
                    )

                    pbar.update(1)

        return self

    def __forward(self, x: np.ndarray) -> np.ndarray:
        logits = self.__linear.forward(x)
        return softmax(logits)


In [40]:
X_train = flatten(normalize(X_train))
X_val = flatten(normalize(X_val))
X_test = flatten(normalize(X_test))

In [41]:
from sklearn.ensemble import BaggingClassifier

def create_single_perceptron(lr: float, epochs: int):
    return Model(
        random_state=RANDOM_STATE,
        learning_rate=lr,
        epochs=epochs,
        in_features=X_train.shape[1],
    )

def create_ensemble(
    lr: float,
    max_samples: float,
    max_features: float,
    epochs: int,
    n_estimators: int
):
    in_features = int(X_train.shape[1] * max_features)

    return BaggingClassifier(
        estimator=Model(
            in_features=in_features,
            learning_rate=lr,
            epochs=epochs,
            random_state=RANDOM_STATE
        ),
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=max_features,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


In [42]:
from sklearn.metrics import roc_auc_score, accuracy_score

def score_model(clf):
    probas_val = clf.predict_proba(X_val)
    y_pred_val = clf.predict(X_val)
    accuracy_val = accuracy_score(y_val, y_pred_val)
    roc_auc_val_list = roc_auc_score(y_val, probas_val, multi_class='ovr', average=None)

    probas_test = clf.predict_proba(X_test)
    y_pred_test = clf.predict(X_test)
    accuracy_test = accuracy_score(y_test, y_pred_test)
    roc_auc_test_list = roc_auc_score(y_test, probas_test, multi_class='ovr', average=None)

    return {
        "accuracy_val": accuracy_val,
        "accuracy_test": accuracy_test,
        "min_roc_auc_val": float(roc_auc_val_list.min()),
        "avg_roc_auc_val": float(roc_auc_val_list.mean()),
        "max_roc_auc_val": float(roc_auc_val_list.max()),
        "min_roc_auc_test": float(roc_auc_test_list.min()),
        "avg_roc_auc_test": float(roc_auc_test_list.mean()),
        "max_roc_auc_test": float(roc_auc_test_list.max()),
    }


In [43]:
from sklearn.neighbors import KNeighborsClassifier

best_models = {
    "knn_accuracy": KNeighborsClassifier(
        n_neighbors=4,
        weights="distance",
        p=2,
        algorithm="auto",
    ),
    "knn_roc_auc": KNeighborsClassifier(
        n_neighbors=36,
        weights="distance",
        p=2,
        algorithm="auto",
    ),
    "single_accuracy": create_single_perceptron(
        lr=0.0011913416,
        epochs=92,
    ),
    "single_min_roc_auc": create_single_perceptron(
        lr=0.0022825994,
        epochs=10
    ),
    "single_avg_roc_auc": create_single_perceptron(
        lr=0.0016008706,
        epochs=40,
    ),
    "ensemble_accuracy": create_ensemble(
        lr=0.10900295,
        max_samples=0.97673649,
        max_features=0.59042108,
        epochs=19,
        n_estimators=47,
    ),
    "ensemble_min_roc_auc": create_ensemble(
        lr=0.0266679893,
        max_samples=0.59587963,
        max_features=0.72146833,
        epochs=12,
        n_estimators=61
    ),
    "ensemble_avg_roc_auc": create_ensemble(
        lr=0.07412715,
        max_samples=0.85040778,
        max_features=0.57341613,
        epochs=21,
        n_estimators=191
    ),
}


In [44]:
results = {}

for name in best_models:
    print(name)
    clf = best_models[name]
    clf.fit(X_train, y_train)
    results[name] = score_model(best_models[name])


knn_accuracy
knn_roc_auc
single_accuracy
single_min_roc_auc
single_avg_roc_auc
ensemble_accuracy
ensemble_min_roc_auc
ensemble_avg_roc_auc


In [52]:
import json

with open("/home/llirik42/PycharmProjects/PerceptronLab/results.json", "w") as f:
    json.dump(results, f)


In [54]:
import pandas as pd

val_metrics = [
    "accuracy_val",
    "min_roc_auc_val",
    "avg_roc_auc_val",
    "max_roc_auc_val",
]

df_val = pd.DataFrame({
    name: {metric.replace("_val", ""): values[metric] for metric in val_metrics}
    for name, values in results.items()
}).T

df_val.index.name = "Model"

test_metrics = [
    "accuracy_test",
    "min_roc_auc_test",
    "avg_roc_auc_test",
    "max_roc_auc_test",
]

df_test = pd.DataFrame({
    name: {metric.replace("_test", ""): values[metric] for metric in test_metrics}
    for name, values in results.items()
}).T

df_test.index.name = "Model"

print("Validation metrics:")
display(df_val.round(8))

print("Test metrics:")
display(df_test.round(8))

Validation metrics:


,accuracy,min_roc_auc,avg_roc_auc,max_roc_auc
Model,,,,
knn_accuracy,0.971000,0.985808,0.994393,0.999002
knn_roc_auc,0.952917,0.994333,0.997875,0.999888
single_accuracy,0.923333,0.989445,0.993762,0.998424
single_min_roc_auc,0.920667,0.989172,0.993501,0.998368
single_avg_roc_auc,0.922417,0.989395,0.993752,0.998417
ensemble_accuracy,0.925417,0.990506,0.994283,0.998494
ensemble_min_roc_auc,0.923000,0.990379,0.994285,0.998392
ensemble_avg_roc_auc,0.925500,0.990744,0.994684,0.998538


Test metrics:


,accuracy,min_roc_auc,avg_roc_auc,max_roc_auc
Model,,,,
knn_accuracy,0.9693,0.988396,0.993808,0.998668
knn_roc_auc,0.9541,0.994417,0.997728,0.999249
single_accuracy,0.9227,0.988263,0.993869,0.999041
single_min_roc_auc,0.9206,0.987877,0.993653,0.999123
single_avg_roc_auc,0.9227,0.988144,0.993847,0.999082
ensemble_accuracy,0.9277,0.990339,0.994750,0.999047
ensemble_min_roc_auc,0.9264,0.990665,0.994688,0.999145
ensemble_avg_roc_auc,0.9274,0.990985,0.995020,0.999111


In [47]:
# import optuna
# from sklearn.metrics import roc_auc_score

In [48]:
# from sklearn.neighbors import KNeighborsClassifier
#
# def objective(trial):
#     k = trial.suggest_int('k', 1, 50)
#     weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
#     p = trial.suggest_int('p', 1, 2)
#
#     clf = KNeighborsClassifier(
#         p=p,
#         algorithm="auto",
#         n_neighbors=k,
#         weights=weights,
#         n_jobs=-1,
#         leaf_size=30,
#     )
#
#     clf.fit(X_train, y_train)
#
#     probas = clf.predict_proba(X_val)
#     score = roc_auc_score(y_val, probas, multi_class="ovr", average="macro")
#
#     return score
#
# study = optuna.create_study(direction='maximize')
#
# study.optimize(objective, n_trials=50)


In [49]:
# def objective(trial):
#     learning_rate = trial.suggest_float('learning_rate', 0.0005, 0.005, log=True)
#     epochs = trial.suggest_int('epochs', 5, 40)
#
#     model = Model(learning_rate=learning_rate, epochs=epochs, in_features=784)
#     model.fit(X_train, y_train)
#
#     probas = model.predict_proba(X_val)
#     score = roc_auc_score(y_val, probas, multi_class="ovr", average="macro")
#
#     return score
#
# # Создаём исследование
# study = optuna.create_study(direction='maximize')
#
# # Запускаем оптимизацию
# study.optimize(objective, n_trials=100)


In [50]:
# from sklearn.ensemble import BaggingClassifier
#
# def objective(trial):
#     learning_rate = trial.suggest_float('learning_rate', 0.005, 0.075, log=True)
#     max_samples = trial.suggest_float('max_samples', 0.25, 0.9)
#     max_features = trial.suggest_float('max_features', 0.5, 1.0)
#     epochs = trial.suggest_int('epochs', 5, 25)
#     n_estimators = trial.suggest_int('n_estimators', 15, 200)
#
#     in_features = int(X_train.shape[1] * max_features)
#
#     model = BaggingClassifier(
#         estimator=Model(
#             in_features=in_features,
#             learning_rate=learning_rate,
#             epochs=epochs,
#             random_state=RANDOM_STATE
#         ),
#         n_estimators=n_estimators,
#         max_samples=max_samples,
#         max_features=max_features,
#         bootstrap=True,
#         random_state=RANDOM_STATE,
#         n_jobs=-1
#     )
#     model.fit(X_train, y_train)
#
#     probas = model.predict_proba(X_val)
#     score = roc_auc_score(y_val, probas, multi_class="ovr", average="macro")
#
#     return score
#
# # Создаём исследование
# study = optuna.create_study(direction='maximize')
#
# # Запускаем оптимизацию
# study.optimize(objective, n_trials=100)
